# jax.grad

In [1]:
import jax
import jax.numpy as jnp

## Basic Usage

`jax.grad(f)` returns a new function that computes the gradient of `f` with respect to its **first argument**.

Rules:
- Input must be a float (or float array)
- Output must be a scalar

In [2]:
# f(x) = x^2  =>  f'(x) = 2x
def f(x):
    return x ** 2

grad_f = jax.grad(f)

print(grad_f(3.0))   # 2 * 3 = 6.0
print(grad_f(5.0))   # 2 * 5 = 10.0

6.0
10.0


## Gradients of Arrays

When the input is an array, `grad` returns a gradient array of the same shape (element-wise partial derivatives).

In [3]:
# g(w) = sum(w^2)  =>  dg/dw_i = 2 * w_i
def g(w):
    return jnp.sum(w ** 2)

w = jnp.array([1.0, 2.0, 3.0])
print(jax.grad(g)(w))   # [2., 4., 6.]

[2. 4. 6.]


## argnums

By default `grad` differentiates with respect to argument `0`. Use `argnums` to target a different argument or multiple arguments at once.

In [4]:
def h(x, y):
    return x ** 2 + x * y + y ** 3

# dh/dy = x + 3y^2
grad_y = jax.grad(h, argnums=1)
print(grad_y(2.0, 3.0))   # 2 + 3*9 = 29.0

# differentiate with respect to both x and y
grad_xy = jax.grad(h, argnums=(0, 1))
print(grad_xy(2.0, 3.0))  # dh/dx = 2x+y = 7,  dh/dy = 29

29.0
(Array(7., dtype=float32, weak_type=True), Array(29., dtype=float32, weak_type=True))


## value_and_grad

Returns both the function value and its gradient in a single forward pass — useful during training when you need both the loss and the gradients.

In [5]:
def loss(w):
    return jnp.sum(w ** 2)

w = jnp.array([1.0, 2.0, 3.0])
val, grads = jax.value_and_grad(loss)(w)

print("loss :", val)    # 1 + 4 + 9 = 14.0
print("grads:", grads)  # [2., 4., 6.]

loss : 14.0
grads: [2. 4. 6.]


## Higher-Order Derivatives

`jax.grad` can be composed with itself to compute second- or higher-order derivatives.

In [6]:
# f(x) = x^4
# f'(x) = 4x^3
# f''(x) = 12x^2
# f'''(x) = 24x

def f(x):
    return x ** 4

df   = jax.grad(f)
ddf  = jax.grad(df)
dddf = jax.grad(ddf)

x = 2.0
print(f"f(x)    = {f(x)}")     # 16
print(f"f'(x)   = {df(x)}")    # 32
print(f"f''(x)  = {ddf(x)}")   # 48
print(f"f'''(x) = {dddf(x)}")  # 48

f(x)    = 16.0
f'(x)   = 32.0
f''(x)  = 48.0
f'''(x) = 48.0


## Gradients of Pytrees

`jax.grad` works with any JAX pytree (dict, list, nested structure) — the same shape as typical neural network params.

In [7]:
def model_loss(params, x):
    # simple linear model: loss = (w*x + b)^2
    pred = params["w"] * x + params["b"]
    return pred ** 2

params = {"w": 2.0, "b": 1.0}
x = 3.0

grads = jax.grad(model_loss)(params, x)
print("dL/dw:", grads["w"])   # d/dw (wx+b)^2 = 2(wx+b)*x = 2*7*3 = 42
print("dL/db:", grads["b"])   # d/db (wx+b)^2 = 2(wx+b)   = 2*7   = 14

dL/dw: 42.0
dL/db: 14.0


## Gradient Descent

Minimizing $f(x) = (x - 3)^2$ with gradient descent. The minimum is at $x = 3$.

In [8]:
def objective(x):
    return (x - 3.0) ** 2

grad_obj = jax.grad(objective)

x = 0.0   # starting point
lr = 0.1

for step in range(20):
    g = grad_obj(x)
    x = x - lr * g
    if (step + 1) % 5 == 0:
        print(f"step {step+1:2d} | x = {x:.4f} | f(x) = {objective(x):.6f}")

step  5 | x = 2.0170 | f(x) = 0.966368
step 10 | x = 2.6779 | f(x) = 0.103763
step 15 | x = 2.8944 | f(x) = 0.011141
step 20 | x = 2.9654 | f(x) = 0.001196
